# ScaleRAG – Multimodal RAG Retrieval Pipeline (v1)

This notebook builds the first version of our **multimodal Retrieval-Augmented Generation (RAG)** pipeline using the preprocessed chunks from the data preparation stage. We load the merged text and image chunks and compute dense embeddings for retrieval.

A **SentenceTransformer model** (`all-MiniLM-L6-v2`) produces **384-dimensional embeddings** for text, while **OpenAI’s CLIP model** (`ViT-B/32`) produces **512-dimensional embeddings** for images.  
When a chunk has both an image and caption, we concatenate the normalized caption and image vectors to form an **896-dimensional embedding**.  
All embeddings are normalized and saved, and **FAISS indexes** are built for fast similarity search.  

---

## Pipeline Overview

1. **Model Setup:**  
   Initialize the SentenceTransformer for text and the CLIP model (with processor) for images, and configure the compute device (GPU/CPU).

2. **Load Chunks:**  
   Read all preprocessed RAG chunk files from `data/rag_chunks/*.json` into a single list.

3. **Separate by Type:**  
   Split the merged chunks into text chunks (*paragraph, text, equation*) and image chunks (*figures, tables*).

4. **Embed Text Chunks:**  
   Batch-encode all text contents using the text model, normalize the 384-D vectors, and attach them to chunk records.

5. **Embed Image Chunks:**  
   For each figure/table, use CLIP to extract a 512-D image feature, normalize it, and if a caption exists, encode it with the text model and concatenate (resulting in 896-D). Attach the combined embeddings.

6. **Save Embeddings:**  
   Store the list of embedded chunks to disk in both JSON and Pickle formats for reuse.

7. **Build FAISS Indexes:**  
   Collect text and image vectors into NumPy arrays, create **FAISS IndexFlatIP** indexes (inner-product for cosine similarity), add the vectors, and save the indexes to disk.

8. **Retrieval Demo:**  
   Define query functions to embed user queries and retrieve the top-k relevant text or image chunks via the FAISS indexes.

---

## Output Directory Summary

- `data/RAG/embeddings/all_papers.embeddings.json` – JSON file of all chunk records with embeddings  
- `data/RAG/embeddings/all_papers.embeddings.pkl` – Pickle file of the same embedding data  
- `data/RAG/indexes/text.index.faiss` – FAISS index for text embeddings (384-D)  
- `data/RAG/indexes/image.index.faiss` – FAISS index for image embeddings (896-D)



## Model and Tokenizer Setup (CLIP & SentenceTransformer)

- **SentenceTransformer:**  
  Uses `all-MiniLM-L6-v2` for 384-dimensional text embeddings.

- **CLIP Model:**  
  Loads `openai/clip-vit-base-patch32` and its processor for 512-dimensional image feature extraction.

- **Device Configuration:**  
  Sets `device = cuda` if available, otherwise `cpu`. The CLIP model is moved to this device.  
  The text model remains on CPU in this notebook (but can also be moved to GPU if desired).

In [1]:
import torch
from PIL import Image
# Text model (SentenceTransformer)
from sentence_transformers import SentenceTransformer
text_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Vision model (CLIP)
from transformers import CLIPProcessor, CLIPModel
clip_model_name = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_model_name)
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)

# Ensure models are on CPU or GPU as available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
clip_model = clip_model.to(device)
text_model = text_model


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


## Loading Preprocessed RAG Chunks

In this step, we load all preprocessed chunk files created during the data preparation phase.

1. **Load JSON Files:**  
   Read all chunk files from `data/rag_chunks/*.json`.  
   Each file contains a list of chunk records with fields such as `id`, `type`, `content`, and `metadata`.

2. **Merge Lists:**  
   Combine the contents of all files into a single list named `merged_chunks`.  
   Print the total number of chunks and display one example record to confirm the format.


In [42]:
import json
import glob

merged_chunks = []

for file_path in glob.glob("data/rag_chunks_v2/*.json"):
    with open(file_path, "r") as f:
        file_chunks = json.load(f)
        merged_chunks.extend(file_chunks)

print("Total merged chunks:", len(merged_chunks))
print("First example chunk:")
print(merged_chunks[0])


Total merged chunks: 2330
First example chunk:
{'id': '2406.03243_paragraph_4', 'type': 'paragraph', 'content': 'Inference serving of LLMs plays a key role in LLMpowered services, becoming a critical workload in datacenters. Such services are typically backed by multiple instances of the LLM deployed on a GPU cluster. The system involves a scheduler and an inference engine , where a request is first dispatched by the scheduler to a model serving instance, then gets executed by the inference engine inside. The requests are typically batched for execution on each instance to increase throughput and cost efficiency.', 'metadata': {'page': 1, 'section': 'Llumnix: Dynamic Scheduling for Large Language Model Serving', 'bbox': [317.88, 432.33, 559.747, 536.523]}, 'source_ids': ['2406.03243_paragraph_4']}


## Separating Text and Image Chunks

In this step, we categorize the merged RAG chunks based on their content type.

1. **Filter by Type:**  
   Create two separate lists from `merged_chunks`:  
   - `text_chunks` → where `type` is `"paragraph"`, `"text"`, or `"equation"`  
   - `image_chunks` → where `type` is `"figure"` or `"table"`

2. **Sanity Check:**  
   Print the counts of text vs. image chunks to verify the split.  
   This ensures that each chunk type will later be embedded using the correct model (text or image encoder).


In [3]:
import numpy as np
from torch.nn.functional import normalize

embedded_chunks = []  # final list of chunk dicts with embeddings

text_chunks = [ch for ch in merged_chunks if ch['type'] in ['paragraph', 'text', 'equation']]
image_chunks = [ch for ch in merged_chunks if ch['type'] in ['figure', 'table']]

print(f"📄 Text chunks: {len(text_chunks)} | 🖼️ Image chunks: {len(image_chunks)}")

📄 Text chunks: 1960 | 🖼️ Image chunks: 370


In [4]:
len(text_chunks), len(image_chunks)

(1960, 370)

In [5]:
text_chunks[0]

{'id': '2406.03243_paragraph_4',
 'type': 'paragraph',
 'content': 'Inference serving of LLMs plays a key role in LLMpowered services, becoming a critical workload in datacenters. Such services are typically backed by multiple instances of the LLM deployed on a GPU cluster. The system involves a scheduler and an inference engine , where a request is first dispatched by the scheduler to a model serving instance, then gets executed by the inference engine inside. The requests are typically batched for execution on each instance to increase throughput and cost efficiency.',
 'metadata': {'page': 1,
  'section': 'Llumnix: Dynamic Scheduling for Large Language Model Serving',
  'bbox': [317.88, 432.33, 559.747, 536.523]},
 'source_ids': ['2406.03243_paragraph_4']}

## Embedding Text Chunks

In this step, we compute dense embeddings for all textual RAG chunks using the SentenceTransformer model.

1. **Batch Encoding:**  
   Extract all text contents from `text_chunks` and encode them in batches using:  
   ```python
   text_model.encode(texts, batch_size=32)
   ```  
   The output is a NumPy array of shape *(num_text_chunks, 384)*, where each row represents a 384-dimensional embedding vector.

2. **Normalization:**  
   Normalize each 384-D vector by dividing it by its L2 norm so that all vectors lie on the unit sphere.  
   This ensures that cosine similarity can be computed directly using the inner product.

3. **Attach to Chunks:**  
   For each text chunk, add the normalized embedding (converted to a Python list) under the key `"embedding"`.  
   The updated record structure becomes:  
    ```python
   {
       "id": ...,
       "type": ...,
       "content": ...,
       "metadata": ...,
       "embedding": [...]
   }
   ```

In [6]:
# Batch compute text embeddings for all text chunks to speed up
texts = [ch['content'] for ch in text_chunks]
text_embeds = text_model.encode(texts, batch_size=32, show_progress_bar=True)
# text_embeds will be a numpy array of shape (len(text_chunks), 384) in this case

# Normalize text embeddings
text_embeds = text_embeds / np.linalg.norm(text_embeds, axis=1, keepdims=True)

# Assign back the text embeddings to their chunks
for ch, vec in zip(text_chunks, text_embeds):
    vec_list = vec.tolist()
    ch_emb = vec_list  # 384-dim text embedding
    # we handle that in the image loop instead.
    embedded_chunks.append({
        "id": ch["id"],
        "type": ch["type"],
        "content": ch["content"],
        "metadata": ch.get("metadata", {}).copy(),
        "embedding": ch_emb
    })

print(f"Text embeddings computed for {len(text_chunks)} chunks.")

Batches:   0%|          | 0/62 [00:00<?, ?it/s]

Text embeddings computed for 1960 chunks.


## Embedding Image Chunks (Figures/Tables)

In this step, we generate embeddings for all image-based RAG chunks such as figures and tables.

1. **Open Image:**  
   For each chunk in `image_chunks`, open the associated image file using PIL:  
   ```python
   image = Image.open(path).convert("RGB")
   ```  
   Skip any chunks where the image cannot be loaded or the file path is missing.

2. **CLIP Encoding:**  
   Preprocess each image using the CLIP processor and extract image features.
   This produces a **512-dimensional image feature vector**.

3. **Normalize Image Vector:**  
   Divide the 512-D image vector by its L2 norm to ensure unit length.

4. **Caption Encoding (if present):**  
   If the chunk’s content contains a caption text, encode it using the text model to obtain a **384-D vector**, then normalize it.

5. **Combine Features:**  
   - If the two vectors have matching shapes (rare case), average them.  
   - Otherwise, concatenate the 384-D caption vector and 512-D image vector to form an **896-D combined embedding**.

6. **Attach to Chunks:**  
   Add the combined 896-D embedding (converted to a Python list) under the key `"embedding"`.  
   The final record structure becomes:  
   ```python
   {
       "id": ...,
       "type": ...,
       "content": ...,
       "metadata": ...,
       "embedding": [...]
   }
   ```


In [7]:
# Now handle image-containing chunks (figures, tables)
for ch in image_chunks:
    img_path = ch.get("metadata", {}).get("image_path") or ch.get("image_path")
    try:
        image = Image.open(img_path).convert("RGB")
    except Exception as e:
        print(f"Warning: could not open image at {img_path}: {e}")
        continue  # skip if image not available
    # Preprocess image for CLIP
    inputs = clip_processor(images=image, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device)
    # Get CLIP image feature (512-dim)
    with torch.no_grad():
        image_feat = clip_model.get_image_features(pixel_values=pixel_values)
    image_vec = image_feat.cpu().numpy().flatten()
    # Normalize image embedding
    image_vec = image_vec / np.linalg.norm(image_vec)

    # Check if there's associated text (e.g., a caption in content)
    combined_vec = image_vec
    if ch.get("content"):
        caption = str(ch["content"])
        # Use the same text model for caption
        caption_emb = text_model.encode([caption])[0]
        caption_emb = caption_emb / np.linalg.norm(caption_emb)
        # Concatenate or average with image_vec:
        if caption_emb.shape[0] == image_vec.shape[0]:
            # If by chance using CLIP text encoder for caption, shapes align (512 each)
            combined_vec = (image_vec + caption_emb) / 2.0  # average
        else:
            # Different dimensions (e.g., 384 vs 512), concatenate
            combined_vec = np.concatenate([caption_emb, image_vec])
            # (combined_vec is now 896-dim in this scenario)
            # We could optionally reduce dimension or keep as is for indexing.
    else:
        # No caption text, combined_vec stays as image_vec
        pass

    embedded_chunks.append({
        "id": ch["id"],
        "type": ch["type"],
        "content": ch.get("content", ""),  # might be caption or empty
        "metadata": ch.get("metadata", {}).copy(),
        "embedding": combined_vec.tolist()
    })

print(f"Computed embeddings for {len(embedded_chunks)} chunks.")
# Show an example of a text chunk and an image chunk embedding (truncated for display)
for ex in embedded_chunks[:2]:
    print(f"{ex['type']} chunk '{ex['id']}' -> embedding length {len(ex['embedding'])}, sample: {ex['embedding'][:5]}")

Computed embeddings for 2330 chunks.
paragraph chunk '2406.03243_paragraph_4' -> embedding length 384, sample: [-0.05061816796660423, -0.051298320293426514, -0.03766610100865364, 0.007535737007856369, 0.0434429831802845]
paragraph chunk '2406.03243_paragraph_8' -> embedding length 384, sample: [-0.06758330017328262, 0.048984672874212265, 0.0064912885427474976, 0.07365099340677261, -0.04049195721745491]


## Saving Embeddings to Disk

In this step, we save the computed embeddings for future use.

1. **Create Directory:**  
   Ensure the folder `data/RAG/embeddings` exists.

2. **Save JSON:**  
   Dump the `embedded_chunks` list to `all_papers.embeddings.json`.

3. **Save Pickle:**  
   Write the same list to `all_papers.embeddings.pkl` using `pickle.dump`.

4. **Verification:**  
   Print confirmation messages after saving to confirm that both files were successfully created.


In [8]:
import os, json, pickle

# Create the folder hierarchy
base_dir = "data/RAG/Version_V2"
emb_dir = os.path.join(base_dir, "embeddings")
os.makedirs(emb_dir, exist_ok=True)

print(f" +++ Folder created at: {emb_dir}")

# Save embeddings
json_path = os.path.join(emb_dir, "all_papers.embeddings.json")
pkl_path = os.path.join(emb_dir, "all_papers.embeddings.pkl")

# Save JSON
with open(json_path, "w") as f:
    json.dump(embedded_chunks, f)
print(f" +++ JSON embeddings saved to {json_path}")

# Save Pickle
with open(pkl_path, "wb") as f:
    pickle.dump(embedded_chunks, f)
print(f" +++ Pickle embeddings saved to {pkl_path}")


 +++ Folder created at: data/RAG/Version_V2/embeddings
 +++ JSON embeddings saved to data/RAG/Version_V2/embeddings/all_papers.embeddings.json
 +++ Pickle embeddings saved to data/RAG/Version_V2/embeddings/all_papers.embeddings.pkl


## Loading and Verifying Saved Embeddings

In this step, we perform a quick sanity check to ensure that the saved embedding files are valid and correctly structured.

1. **Load Pickle:**  
   Reload the embeddings list from `all_papers.embeddings.pkl`.

2. **Inspect Structure:**  
   Print the total number of chunks, view example keys from the first record (`id`, `type`, `content`, etc.), and check the embedding dimensions.

3. **Verification:**  
   Confirm that text chunks have **384-D embeddings** and image chunks have **896-D embeddings**.


In [1]:
import pickle

with open("../data2/RAG/Version_V2/embeddings/all_papers.embeddings.pkl", "rb") as f:
    embedded_chunks = pickle.load(f)

print("Loaded", len(embedded_chunks), "chunks.")
print("Example record keys:", embedded_chunks[0].keys())
print("Example metadata:", embedded_chunks[0]["metadata"])
print("Embedding dim:", len(embedded_chunks[0]["embedding"]))

Loaded 75431 chunks.
Example record keys: dict_keys(['id', 'type', 'content', 'metadata', 'embedding'])
Example metadata: {'page': 1, 'section': 'KEYWORDS', 'bbox': [53.484, 264.288, 295.032, 287.638]}
Embedding dim: 384


In [3]:
import numpy as np

text_dims = [len(ch["embedding"]) for ch in embedded_chunks if ch["type"] in ["text","paragraph","equation"]]
image_dims = [len(ch["embedding"]) for ch in embedded_chunks if ch["type"] in ["figure","table"]]

print(f"Text embeddings: mean {np.mean(text_dims):.0f} ± {np.std(text_dims):.1f}")
print(f"Image embeddings: mean {np.mean(image_dims):.0f} ± {np.std(image_dims):.1f}")

Text embeddings: mean 384 ± 0.0
Image embeddings: mean 896 ± 0.0


## Splitting Embeddings for Indexing

In this step, we separate the embeddings by type to prepare for FAISS indexing.

1. **Filter by Dimension:**  
   Iterate over all loaded `embedded_chunks`.  
   - If the type is a text type and the embedding length is **384**, append it to `text_records` and collect its vector.  
   - If the type is an image/table and the embedding length is **896**, append it to `image_records` and collect its vector.  
   - Ignore any records with unexpected embedding dimensions.

2. **Stack Vectors:**  
   Convert the collected vectors into NumPy arrays:  
   - `text_vectors` → shape `(N_text, 384)`  
   - `image_vectors` → shape `(N_image, 896)`

3. **Print Shapes:**  
   Display the number of records and array shapes to verify the data integrity before indexing.


## Building and Saving FAISS Indexes

1. **Create Indexes:**  
   Initialize FAISS indexes using **inner product (IP)** similarity, which corresponds to cosine similarity when embeddings are normalized.  
   - `index_text = IndexFlatIP(384)` for text embeddings  
   - `index_image = IndexFlatIP(896)` for image embeddings

2. **Add Vectors:**  
   Add `text_vectors` to `index_text` and `image_vectors` to `index_image`.  
   The final index sizes should match the number of added vectors.

3. **Save to Disk:**  
   Ensure the folder `data/RAG/indexes` exists.  
   Save the indexes as `text.index.faiss` and `image.index.faiss`, and print confirmation messages to verify successful writes.


In [4]:
import numpy
import faiss
print("NumPy version:", numpy.__version__)
print("FAISS version:", faiss.__version__)

NumPy version: 1.26.4
FAISS version: 1.7.2


In [5]:
import numpy as np
import faiss

# --- Split into text chunks (384-D) and figure/table chunks (896-D)

text_records = []
text_vectors = []

image_records = []
image_vectors = []

for ch in embedded_chunks:
    vec = np.array(ch["embedding"], dtype="float32")
    dim = vec.shape[0]

    if ch["type"] in ["text", "paragraph", "equation", "table_caption", "figure_caption"]:
        # safety check: only keep consistent dim=384
        if dim == 384:
            text_records.append(ch)
            text_vectors.append(vec)
    elif ch["type"] in ["figure", "table"]:
        # safety check: only keep consistent dim=896 (caption+image concat case)
        if dim == 896:
            image_records.append(ch)
            image_vectors.append(vec)
    else:
        # ignore other types for now or print to debug
        pass

text_vectors = np.stack(text_vectors, axis=0) if len(text_vectors) > 0 else np.zeros((0,384), dtype="float32")
image_vectors = np.stack(image_vectors, axis=0) if len(image_vectors) > 0 else np.zeros((0,896), dtype="float32")

print("Text records:", len(text_records), "| text_vectors shape:", text_vectors.shape)
print("Image records:", len(image_records), "| image_vectors shape:", image_vectors.shape)

# --- Build FAISS indexes

dim_text = text_vectors.shape[1]
dim_image = image_vectors.shape[1]

index_text = faiss.IndexFlatIP(dim_text)    # cosine sim if vectors are normalized
index_image = faiss.IndexFlatIP(dim_image)  # cosine sim if vectors are normalized

index_text.add(text_vectors)
index_image.add(image_vectors)

print("FAISS index_text size:", index_text.ntotal)
print("FAISS index_image size:", index_image.ntotal)

# Save indexes to disk so you don't have to rebuild next time
import os
os.makedirs("../data2/RAG/Version_V2/indexes", exist_ok=True)

faiss.write_index(index_text, "../data2/RAG/Version_V2/indexes/text.index.faiss")
faiss.write_index(index_image, "../data2/RAG/Version_V2/indexes/image.index.faiss")

print(" +++ Saved FAISS indexes.")


Text records: 1960 | text_vectors shape: (1960, 384)
Image records: 370 | image_vectors shape: (370, 896)
FAISS index_text size: 1960
FAISS index_image size: 370
 +++ Saved FAISS indexes.
